In [2]:
import numpy as np
import pandas as pd

In [4]:
temp_df = pd.read_csv('IMDB Dataset.csv')

In [5]:
df = temp_df.iloc[:10000]

In [6]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
import re 
def remove_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'', text)

In [8]:
df['review'] = df['review'].apply(remove_tags)

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_51964\2336150696.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(remove_tags)


In [9]:
df['review'] = df['review'].str.lower()

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_51964\724319867.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].str.lower()


In [12]:
import nltk
from nltk.corpus import stopwords
sw = stopwords.words('english')

In [13]:
def remove_stopwords(text):
    new_text = []
    for w in text.split():
        if w in sw:
            new_text.append('')
        else:
            new_text.append(w)
    x = new_text[:]
    new_text.clear()
    return " ".join(x)

In [14]:
df['review'] = df['review'].apply(remove_stopwords)

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_51964\3992406652.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(remove_stopwords)


In [15]:
import gensim

In [16]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [17]:
story = []
for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))

In [19]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=2
)

In [20]:
model.build_vocab(story)

In [21]:
model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

(5482653, 5910915)

In [22]:
len(model.wv.index_to_key)

35249

In [24]:
# by now, we have vectors for each word
# we'll create vector for each review (document)

In [26]:
def document_vector(doc):
    # remove out-of-vocabulary words
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis=0)

In [29]:
# to get the first review converted into vector that has 100 dimensions
document_vector(df['review'].values[0])

array([-0.33463773,  0.34564573, -0.01706579,  0.3072654 ,  0.01175327,
       -0.5635027 ,  0.20955156,  0.77290416, -0.21175227, -0.27429408,
       -0.1920948 , -0.35186878, -0.13238715,  0.3628957 ,  0.19055307,
       -0.36195514,  0.1044424 , -0.5198191 ,  0.01718005, -0.6813829 ,
        0.35710979,  0.3619696 ,  0.31546834, -0.22647396,  0.05731743,
        0.01489431, -0.30253974, -0.03944266, -0.35124466,  0.09488483,
        0.5990585 ,  0.01895441,  0.10045303, -0.445893  , -0.19037169,
        0.59648216,  0.19270614, -0.3917916 , -0.2998935 , -0.6718374 ,
        0.02527132, -0.37429038, -0.22272478, -0.09896593,  0.18456939,
       -0.13355142, -0.31984183, -0.09537822,  0.32999775,  0.21585104,
        0.2351134 , -0.40787637, -0.2144673 , -0.05057793, -0.29443642,
        0.10156656,  0.2882041 , -0.11690387, -0.2925037 ,  0.09374748,
        0.12917224,  0.10888202, -0.07070465, -0.05298241, -0.38922268,
        0.39043987,  0.12381735,  0.17321578, -0.5675028 ,  0.34

In [31]:
from tqdm import tqdm

In [32]:
X = []
for doc in tqdm(df['review'].values):
    X.append(document_vector(doc))

100%|█████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [07:17<00:00, 22.85it/s]


In [33]:
X = np.array(X)

In [34]:
X.shape

(10000, 100)

In [35]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

y = encoder.fit_transform(df['sentiment'])

In [36]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [37]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [38]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test, y_pred)

0.8015